In [1]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.7 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [3]:
file_path = "Ex4_Dataset.txt"

with open(file_path, "r", encoding="utf-8") as file:
    documents = file.readlines()

# Remove empty lines
documents = [doc.strip() for doc in documents if doc.strip()]

print("Dataset loaded successfully!")
print(f"Total text chunks: {len(documents)}")

Dataset loaded successfully!
Total text chunks: 7


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [5]:
doc_embeddings = model.encode(
    documents,
    convert_to_numpy=True
)

print("Embeddings created successfully!")
print("Embedding shape:", doc_embeddings.shape)

Embeddings created successfully!
Embedding shape: (7, 384)


In [6]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(doc_embeddings)

print("FAISS index created successfully!")
print("Number of vectors in index:", index.ntotal)

FAISS index created successfully!
Number of vectors in index: 7


In [7]:
query = input("Enter your search query: ")

query_embedding = model.encode(
    [query],
    convert_to_numpy=True
)

print("Query converted into an embedding!")

Enter your search query: How does AI learn from data?
Query converted into an embedding!


In [8]:
top_k = 3

distances, indices = index.search(query_embedding, top_k)

print("Search completed!")

Search completed!


In [9]:
for rank, idx in enumerate(indices[0]):
    print(f"\nRank {rank + 1}:")
    print(documents[idx])
    print(f"Distance: {distances[0][rank]}")


Rank 1:
Machine learning allows computers to learn patterns from data and make predictions.
Distance: 0.7153877019882202

Rank 2:
Generative AI can create new content such as text, images, audio, and code.
Distance: 0.9217164516448975

Rank 3:
Artificial intelligence enables machines to perform tasks that normally require human intelligence.
Distance: 0.9380990266799927


In [10]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 75.2 MB/s eta 0:00:00


In [11]:
%%writefile app.py

import streamlit as st
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Page title
st.title("Semantic Search App")
st.write("Search the dataset using semantic similarity.")

# Load dataset
file_path = "Ex4_Dataset.txt"

with open(file_path, "r", encoding="utf-8") as file:
    documents = file.readlines()

documents = [doc.strip() for doc in documents if doc.strip()]

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create document embeddings
doc_embeddings = model.encode(
    documents,
    convert_to_numpy=True
)

# Create FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# Search box
query = st.text_input("Enter your search query:")

# Number of results
top_k = 3

if st.button("Search"):
    if query:
        # Convert query to embedding
        query_embedding = model.encode(
            [query],
            convert_to_numpy=True
        )

        # Search FAISS
        distances, indices = index.search(
            query_embedding,
            top_k
        )

        # Display results
        st.subheader("Search Results")

        for rank, idx in enumerate(indices[0]):
            st.write(f"### Rank {rank + 1}")
            st.write(documents[idx])
            st.write(f"Distance: {distances[0][rank]}")
    else:
        st.warning("Please enter a search query.")

Writing app.py


In [12]:
!streamlit run app.py &>/content/logs.txt &

In [19]:
from google.colab import output

output.serve_kernel_port_as_iframe(8502, height=800)

<IPython.core.display.Javascript object>